# One PINN experiment
This notebook defines and launches exactly one experiment. Change the configuration cell, then run the training cell once.

In [ ]:
!pip install -q wandb accelerate

In [ ]:
import os

base_path = '/kaggle/working'
project_name = 'weight-vs-loss-convergence-pinn'
project_root = os.path.join(base_path, project_name)

if not os.path.exists(project_root):
    os.chdir(base_path)
    !git clone https://github.com/nthday-jpg/weight-vs-loss-convergence-pinn.git
else:
    os.chdir(project_root)
    !git pull

os.environ['PYTHONPATH'] = project_root
%cd {project_root}

from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_KEY')

In [ ]:
# Generate data for this experiment.
data_file = os.path.join(project_root, 'navier-stokes', 'data', 'navier_stokes_1d.pt')
!python navier-stokes/data/gen_data.py --save_dir navier-stokes/data --base_name navier_stokes_1d --nx 256 --steps 200 --t_final 0.6 --mu 0.01

In [ ]:
import json

# One configuration produces one W&B run. Keep the architecture shared:
# the three outputs are coupled primitive variables (rho, u, p).
config = {
    'problem': 'navier_stokes',
    'layers': [2, 64, 64, 64, 3],
    'learning_rate': 1e-3,
    'num_epochs': 500,
    'step_per_epoch': 100,
    'batch_size': 2048,
    'seed': 42,
    'balancer_type': 'softadapt',  # use 'uniform' for the fixed baseline
    'softadapt_params': {'beta': 0.1, 'ema_alpha': 0.05},
    'wandb_project': 'loss-weight-convergence',
    'wandb_run_name': 'navier-stokes-softadapt',
}

# Environment variables are inherited by the Accelerate subprocess.
os.environ['PINN_CONFIG'] = json.dumps(config)
os.environ['PINN_DATA_FILE'] = data_file

In [ ]:
# One process keeps adaptive balancer state deterministic and synchronized.
!accelerate launch --num_processes 1 {project_root}/train.py